In [10]:
import os

os.chdir("../")

In [19]:
from dataclasses import dataclass
from pathlib import Path


@dataclass(frozen=True)
class ModelTrainerConfig:
    root_dir: Path
    train: Path
    test: Path
    model_name: str
    params: dict
    target_column: str

In [20]:
import ares.constants as const
from ares.utils.common import read_yaml, create_directories

In [21]:
class ConfigurationManager:
    def __init__(
        self,
        config_filepath=const.CONFIG_FILE_PATH,
        params_filepath=const.PARAMS_FILE_PATH,
        schema_filepath=const.SCHEMA_FILE_PATH,
    ):
        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        self.schema = read_yaml(schema_filepath)

        create_directories([self.config.artifacts_root])

    def get_model_trainer_config(self) -> ModelTrainerConfig:
        config = self.config.model_trainer
        params = self.params.CatBoost
        schema = self.schema.TARGET_COLUMN

        create_directories([config.root_dir])

        model_trainer_config = ModelTrainerConfig(
            root_dir=config.root_dir,
            train=config.train,
            test=config.test,
            model_name=config.model_name,
            target_column=schema.name,
            params=params,
        )

        return model_trainer_config

In [22]:
import os
import pandas as pd
import numpy as np
from joblib import dump
from catboost import CatBoostRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from ares import logger


In [23]:
class ModelTrainer:
    def __init__(self, config: ModelTrainerConfig):
        self.config = config

    def train(self):
        """Train baseline CatBoost and save model.

        Returns
        -------
        model: CatBoostRegressor
        metrics: dict[str, float]
        """
        train = pd.read_csv(self.config.train)
        test = pd.read_csv(self.config.test)

        train_x = train.drop([self.config.target_column], axis=1)
        test_x = test.drop([self.config.target_column], axis=1)
        train_y = train[[self.config.target_column]]
        test_y = test[[self.config.target_column]]

        params = self.config.params if self.config.params else {}
        model = CatBoostRegressor(**params)
        model.fit(train_x, train_y, logging_level="Silent")

        y_pred = model.predict(test_x)
        mae = float(mean_absolute_error(test_y, y_pred))
        rmse = float(np.sqrt(mean_squared_error(test_y, y_pred)))
        r2 = float(r2_score(test_y, y_pred))
        metrics = {"mae": mae, "rmse": rmse, "r2": r2}

        out = os.path.join(self.config.root_dir, self.config.model_name)
        dump(model, out)

        logger.info(f"Model trained. Saved to {out}")
        logger.info(f"MAE={mae:.2f},  RMSE={rmse:.2f},  R²={r2:.4f}")

        return model, metrics

In [24]:
try:
    config = ConfigurationManager()
    model_trainer_config = config.get_model_trainer_config()
    model_trainer = ModelTrainer(config=model_trainer_config)
    model_trainer.train()
except Exception as e:
    raise e

[2026-02-01 12:51:07,170: INFO: common: YAML file: config/config.yaml loaded successfully]
[2026-02-01 12:51:07,175: INFO: common: YAML file: params.yaml loaded successfully]
[2026-02-01 12:51:07,179: INFO: common: YAML file: schema.yaml loaded successfully]
[2026-02-01 12:51:07,182: INFO: common: Created directory at: artifacts]
[2026-02-01 12:51:07,184: INFO: common: Created directory at: artifacts/model_trainer]
[2026-02-01 12:51:09,084: INFO: 1101170724: Model trained. Saved to artifacts/model_trainer/model.joblib]
[2026-02-01 12:51:09,085: INFO: 1101170724: MAE=0.34,  RMSE=0.46,  R²=0.8514]
